In [ ]:
# create synthetic tourist by IPF algorithm
import pandas as pd
import numpy as np
from itertools import product
from ipfn import ipfn

# ----------------------------
# 1. Category and Margin Definitions
# ----------------------------
categories = {
    "gender": ['m', 'f'],
    "age_group": ['10s', '20s', '30s', '40s', '50s', '60+'],
    "companion_type": ['alone', 'partner', 'family & relatives', 'acquaintances', 'groups'],
    "visit_frequency": ['1', '2+']
}

margins = {
    "gender": np.array([563, 1047]),
    "age_group": np.array([21, 76, 110, 218, 394, 795]),
    "companion_type": np.array([399, 466, 433, 300, 16]),
    "visit_frequency": np.array([32, 1581])
}

total_population = 1613

# ----------------------------
# 2. Create initial seed dataframe
# ----------------------------
all_combinations = list(product(
    categories['gender'],
    categories['age_group'],
    categories['companion_type'],
    categories['visit_frequency']
))

df = pd.DataFrame(all_combinations, columns=['gender', 'age_group', 'companion_type', 'visit_frequency'])
df['total'] = 1.0

# ----------------------------
# 3. Margin Series
# ----------------------------
gender_margin = pd.Series(margins['gender'], index=categories['gender'])
age_margin = pd.Series(margins['age_group'], index=categories['age_group'])
companion_margin = pd.Series(margins['companion_type'], index=categories['companion_type'])
freq_margin = pd.Series(margins['visit_frequency'], index=categories['visit_frequency'])

aggregates = [gender_margin, age_margin, companion_margin, freq_margin]
dimensions = [['gender'], ['age_group'], ['companion_type'], ['visit_frequency']]

# ----------------------------
# 4. IPF Correction
# ----------------------------
IPF = ipfn.ipfn(df, aggregates, dimensions)
adjusted_df = IPF.iteration()

# ----------------------------
# 5. Convert to probability → for sampling
# ----------------------------
adjusted_df['prob'] = adjusted_df['total'] / adjusted_df['total'].sum()

# ----------------------------
# 6. Create synthetic population
# ----------------------------
synthetic_population = adjusted_df.sample(
    n=total_population,
    weights='prob',
    replace=True 
).reset_index(drop=True)

# ----------------------------
# 7. Add person ID
# ----------------------------
age_group_to_range = {
    '10s': (10, 19),
    '20s': (20, 29),
    '30s': (30, 39),
    '40s': (40, 49),
    '50s': (50, 59),
    '60+': (60, 80)  
}
synthetic_population['person_id'] = ['t_' + str(i+1) for i in range(total_population)]

# Create age
synthetic_population['age'] = synthetic_population['age_group'].apply(
    lambda g: np.random.randint(age_group_to_range[g][0], age_group_to_range[g][1] + 1)
)

synthetic_population = synthetic_population.drop(columns=['total', 'prob'])

column_order = ['person_id', 'gender', 'age', 'age_group', 'companion_type', 'visit_frequency']
synthetic_population = synthetic_population[column_order]

output_path = "C:/Users/CHOI/Desktop/Temporary working folder/matsim/synthetic population/synthetic_population_10pct.csv"
synthetic_population.to_csv(output_path, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compatibility

print(f"✅ {output_path} save success")


In [ ]:
#data cleaning for ref.person
import pandas as pd
from datetime import timedelta, datetime

df = pd.read_csv(r"C:\Users\CHOI\Desktop\Temporary working folder\matsim\ref_person\raw_data.csv")

midnight = datetime.strptime("00:00", "%H:%M")

df['departure_time'] = df.apply(
    lambda row: (midnight + timedelta(hours=int(row['dep. H']), minutes=int(row['dep. M']))).strftime('%H:%M'),
    axis=1
)
df['arrive_time'] = df.apply(
    lambda row: (midnight + timedelta(hours=int(row['arr. H']), minutes=int(row['arr. M']))).strftime('%H:%M'),
    axis=1
)

df = df.drop(columns=['dep. H', 'dep. M', 'arr. H', 'arr. M'])

df['start_time'] = ''
df['end_time'] = ''

for pid, group in df.groupby('p-ID'):
    idx = group.index.tolist()
    for i, row_idx in enumerate(idx):
        if i == 0:
            df.loc[row_idx, 'end_time'] = df.loc[row_idx, 'departure_time']
        else:
            df.loc[row_idx, 'start_time'] = df.loc[idx[i - 1], 'arrive_time']
            df.loc[row_idx, 'end_time'] = df.loc[row_idx, 'departure_time']

df.to_csv(r"C:\Users\CHOI\Desktop\Temporary working folder\matsim\ref_person\clean_data.csv", index=False)
print(df[['p-ID', 'departure_time', 'arrive_time', 'start_time', 'end_time']])


In [ ]:
#calculate duration time
import pandas as pd
from datetime import datetime

df = pd.read_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/ref_person/clean_data.csv")

df['departure_time'] = pd.to_datetime(df['departure_time'], format='%H:%M').dt.strftime('%H:%M')
df['arrive_time'] = pd.to_datetime(df['arrive_time'], format='%H:%M').dt.strftime('%H:%M')

df['start_time'] = ''
df['end_time'] = ''

for pid, group in df.groupby('p-ID'):
    idxs = group.index.tolist()
    for i, idx in enumerate(idxs):
        if i == 0:
            df.loc[idx, 'end_time'] = df.loc[idx, 'departure_time']
        else:
            df.loc[idx, 'start_time'] = df.loc[idxs[i - 1], 'arrive_time']
            df.loc[idx, 'end_time'] = df.loc[idx, 'departure_time']

# duration calculation
def compute_duration(row):
    if row['start_time'] == '':
        return ''
    fmt = '%H:%M'
    start = datetime.strptime(row['start_time'], fmt)
    end = datetime.strptime(row['end_time'], fmt)
    diff = end - start
    return str(diff)[:-3]  # 'HH:MM:SS' → 'HH:MM'

df['duration'] = df.apply(compute_duration, axis=1)

df.to_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/ref_person/clean_data_endtime.csv", index=False)

print(df[['p-ID', 'departure_time', 'arrive_time','start_time', 'end_time', 'duration']])


In [ ]:
#Extract kyoto station trips
import pandas as pd

activity_df = pd.read_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/ref_person/clean_data_endtime.csv")

# origin code about over one-day trip
excluded_origins = [99, 58, 39, 43]

# Extract ref_id list about over one-day trip
bad_ref_ids = activity_df[activity_df['origin'].isin(excluded_origins)]['ref_id'].unique()

# delect ref_id list
cleaned_activity_df = activity_df[~activity_df['ref_id'].isin(bad_ref_ids)]

cleaned_activity_df.to_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/ref_person/clean_data_endtime_removed_99_58_39_43.csv", index=False)


In [ ]:
#assign activity chain to synthetic tourist population
import pandas as pd
import numpy as np
import random

np.random.seed(43)
random.seed(43)

synthetic_df = pd.read_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/synthetic population/synthetic_population_10pct.csv")
activity_df = pd.read_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/ref_person/clean_data_endtime_removed_99_58_39_43.csv")

# random select ref_id by priority
def match_activity(person, activity_df):
    keys = [
        ["gender", "age_group", "companion_type", "visit_frequency"],
        ["gender", "age_group", "companion_type"],
        ["gender", "age_group"],
        ["age_group"]
    ]

    for key in keys:
        cond = np.ones(len(activity_df), dtype=bool)
        for k in key:
            cond &= (activity_df[k] == person[k])
        matched = activity_df[cond]

        if not matched.empty:
            bad_ref_ids = matched[matched["origin"].isin(excluded_origins)]["ref_id"].unique()
            valid_ref_ids = [rid for rid in matched["ref_id"].unique() if rid not in bad_ref_ids]

            if valid_ref_ids:
                selected_ref_id = np.random.choice(valid_ref_ids)
                return selected_ref_id
    return None

# assign activity chain from selected ref_id to synthetic tourist population 
expanded_results = []

for _, person in synthetic_df.iterrows():
    selected_ref_id = match_activity(person, activity_df)

    if selected_ref_id is not None:
        matched_trips = activity_df[activity_df["ref_id"] == selected_ref_id]

        for _, trip in matched_trips.iterrows():
            person_trip = person.to_dict()
            person_trip["origin"] = trip["origin"]
            person_trip["Trip_purpose"] = trip["Trip_purpose"]
            person_trip["mode"] = trip["mode"]
            person_trip["start_time"] = trip["start_time"]
            person_trip["end_time"] = trip["end_time"]
            person_trip["duration"] = trip["duration"]
            expanded_results.append(person_trip)
    else:
        person_trip = person.to_dict()
        person_trip["origin"] = np.nan
        person_trip["Trip_purpose"] = np.nan
        person_trip["mode"] = np.nan
        person_trip["start_time"] = np.nan
        person_trip["end_time"] = np.nan
        person_trip["duration"] = np.nan
        expanded_results.append(person_trip)


final_df = pd.DataFrame(expanded_results)
final_df.to_csv("C:/Users/CHOI/Desktop/Temporary working folder/matsim/synthetic population/zone_synthetic_population_10pct.csv", index=False)


In [ ]:
#solution UTF-8 issue on GPT data 
import pandas as pd
import ast
import json
import os

input_folder = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/populartimes_20250421-075511"

output_folder = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/gpt"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.lower().endswith('.csv'):
        input_csv = os.path.join(input_folder, filename)
        base_name = os.path.splitext(filename)[0]

        print(f"📂 processing: {filename}")

        try:
            df = pd.read_csv(input_csv, encoding='utf-8')

            def safe_eval(x):
                try:
                    return ast.literal_eval(x)
                except Exception:
                    return None

            df['popular_times'] = df['popular_times'].apply(lambda x: safe_eval(x) if pd.notna(x) else None)
            df['time_spent'] = df['time_spent'].apply(lambda x: safe_eval(x) if pd.notna(x) else None)

            # save JSON 
            json_file = os.path.join(output_folder, f"{base_name}_cleaned.json")
            df.to_json(json_file, orient='records', force_ascii=False, indent=2)
            print(f"save JSON succeed: {json_file}")

            # 4️JSON → DataFrame
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            df_new = pd.DataFrame(data)

            #  save CSV file by UTF-8
            output_csv = os.path.join(output_folder, f"{base_name}_final_utf8.csv")
            df_new.to_csv(output_csv, index=False, encoding='utf-8-sig')
            print(f"save csv succeed: {output_csv}")

        except Exception as e:
            print(f"⚠ error ({filename}): {e}")

print("success")

In [ ]:
#integrate gpt data into one
import pandas as pd
import os
import ast

output_folder = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/gpt"

# CSV file list to integrate
csv_files = [os.path.join(output_folder, f) for f in os.listdir(output_folder) if f.lower().endswith('.csv')]

# intergrate csv into one
df_list = []
for file in csv_files:
    df = pd.read_csv(file, encoding='utf-8-sig')
    df_list.append(df)
combined_df = pd.concat(df_list, ignore_index=True)

def extract_first_time_spent(x):
    if pd.isna(x):
        return None
    try:
        values = ast.literal_eval(x) if isinstance(x, str) else x
        if isinstance(values, list) and len(values) > 0:
            return values[0]  
        return values
    except Exception:
        return None

combined_df['time_spent'] = combined_df['time_spent'].apply(extract_first_time_spent)

# grouping by place_id → average rating, rating_n, time_spent
agg_df = combined_df.groupby('place_id').agg({
    'name': 'first',  
    'lat': 'first',
    'lon': 'first',
    'node_type': 'first',
    'address': 'first',
    'place': 'first',
    'popular_times': 'first', 
    'rating': 'mean',
    'rating_n': 'mean',
    'current_popularity': 'first',  
    'time_spent': 'mean'
}).reset_index()

final_output = os.path.join(output_folder, 'gpt_datacleaning.csv')
agg_df.to_csv(final_output, index=False, encoding='utf-8-sig')

print(f"save succeed → {final_output}")


In [ ]:
#transfer coordinate for MATSim
import pandas as pd
import geopandas as gpd
from pyproj import Transformer

input_path = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/gpt\gpt_datacleaning.csv"
df = pd.read_csv(input_path)

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32653", always_xy=True)

def convert_coords(row):
    x, y = transformer.transform(row['lon'], row['lat'])
    return pd.Series({'x_coord': round(x, 9), 'y_coord': round(y, 8)})

df[['x_coord', 'y_coord']] = df.apply(convert_coords, axis=1)

print(df.head())

output_path = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/gpt\gpt_datacleaning_matsimcoord.csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')


In [ ]:
#assign coordinate of location from GPT data set
import pandas as pd
import numpy as np
from datetime import timedelta

synthetic_df = pd.read_csv(r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/synthetic population/zone_synthetic_population_10pct.csv")
zone_df = pd.read_csv(r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/matsim_tourist_population/with zone_number.csv")

target_origins = [40, 41, 45, 47]
kyoto_x = 569016.31890973
kyoto_y = 3871735.40867188
kyoto_type = 'kyoto_station'

synthetic_df.loc[synthetic_df['origin'].isin(target_origins), 'x'] = kyoto_x
synthetic_df.loc[synthetic_df['origin'].isin(target_origins), 'y'] = kyoto_y
synthetic_df.loc[synthetic_df['origin'].isin(target_origins), 'type'] = kyoto_type

def duration_to_minutes(dur_str):
    try:
        hours, minutes = map(int, str(dur_str).split(':'))
        delta = timedelta(hours=hours, minutes=minutes)
        return delta.total_seconds() // 60  # 분 단위로 반환
    except:
        return np.nan

synthetic_df['duration_min'] = synthetic_df['duration'].apply(duration_to_minutes)

for idx, row in synthetic_df.iterrows():
    if row['origin'] in target_origins:
        continue  

    matching_zone = zone_df[zone_df['zone'] == row['origin']]

    if matching_zone.empty:
        continue

    # Step 1: zone + Trip_purpose + duration
    candidate = matching_zone[
        (matching_zone['poi_type'] == row['Trip_purpose']) &
        (matching_zone['time_spent'] < row['duration_min'])
    ]

    # Step 2: zone + Trip_purpose
    if candidate.empty:
        candidate = matching_zone[
            matching_zone['poi_type'] == row['Trip_purpose']
        ]

    # Step 3: zone
    if candidate.empty:
        candidate = matching_zone

    if candidate.empty:
        continue

    # gravity
    gravity_values = candidate['gravity'].values

    if gravity_values.max() == gravity_values.min():
        selected = candidate.sample()
    else:
        gravity_norm = (gravity_values - gravity_values.min()) / (gravity_values.max() - gravity_values.min() + 1e-6)
        probabilities = gravity_norm / gravity_norm.sum()
        selected = candidate.sample(weights=probabilities)

    poi_type_raw = selected['poi_type'].values[0]
    poi_type_adj = 'leisure' if poi_type_raw in ['sightseeing', 'food'] else poi_type_raw

    key = f"{selected['zone'].values[0]}_{poi_type_adj}_{selected['place_id'].values[0]}"

    if 'place_id_map' not in locals():
        place_id_map = {}
        next_id = 1

    if key not in place_id_map:
        place_id_map[key] = next_id
        next_id += 1
        
    synthetic_df.loc[idx, 'x'] = selected['x_coord'].values[0]
    synthetic_df.loc[idx, 'y'] = selected['y_coord'].values[0]
    synthetic_df.loc[idx, 'type'] = f"{poi_type_adj}"

synthetic_df.to_csv(r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/synthetic population/zone_synthetic_population_10pct_withxy.csv", index=False)


In [ ]:
#create tourist_synthetic population.XML
import pandas as pd
import gzip
from lxml import etree as ET

# Load input CSV
input_file = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/synthetic population/zone_synthetic_population_1pct_withxy.csv"
df = pd.read_csv(input_file)

# Prepare XML root
population = ET.Element("population")

# Add global attributes
population_attributes = ET.SubElement(population, "attributes")
ET.SubElement(population_attributes, "attribute", name="coordinateReferenceSystem", attrib={"class": "java.lang.String"}).text = "EPSG:32653"

# Group by person_id
for person_id, group in df.groupby('person_id'):
    person = ET.SubElement(population, "person", id=str(person_id))

    # Add attributes
    attributes = ET.SubElement(person, "attributes")
    def add_attr(name, clazz, value):
        ET.SubElement(attributes, "attribute", name=name, attrib={"class": clazz}).text = str(value)

    row = group.iloc[0]
    add_attr("gender", "java.lang.String", row['gender'])
    add_attr("age", "java.lang.Integer", row['age'])
    add_attr("companion_type", "java.lang.String", row['companion_type'])
    add_attr("visit_frequency", "java.lang.String", row['visit_frequency'])
    add_attr("subpopulation", "java.lang.String", "person")
    add_attr("gem", "java.lang.Integer", 12065136)
    add_attr("ars", "java.lang.Long", 120650136136)
    add_attr("restricted_mobility", "java.lang.Boolean", "false")
    add_attr("carAvail", "java.lang.String", "always")
    add_attr("bikeAvail", "java.lang.String", "always")
    add_attr("ptAboAvail", "java.lang.String", "never")

    # Create plan
    plan = ET.SubElement(person, "plan", selected="yes")

    for i, (_, act_row) in enumerate(group.iterrows()):
        act_attrib = {
            "type": str(act_row['type']),
            "x": str(act_row['x']),
            "y": str(act_row['y'])
        }
        if i == 0:
            act_attrib["end_time"] = str(act_row['end_time'])
        else:
            act_attrib["start_time"] = str(act_row['start_time'])
            act_attrib["end_time"] = str(act_row['end_time'])
        ET.SubElement(plan, "activity", **act_attrib)

        # Always add leg, including after the last activity
        leg = ET.SubElement(plan, "leg", mode=str(act_row['mode']))
        leg_attr = ET.SubElement(leg, "attributes")
        ET.SubElement(leg_attr, "attribute", name="routingMode", attrib={"class": "java.lang.String"}).text = str(act_row['mode'])

    # Add final activity at kyoto_station (no end_time)
    final_act = ET.SubElement(plan, "activity", type="home", x="569016.3189", y="3871735.409")
    final_attr = ET.SubElement(final_act, "attributes")

# Save with DTD included
output_path = r"C:/Users/CHOI/Desktop/Temporary working folder/matsim/kyoto_tourists_population_1pct.xml"
dtd_str = '<!DOCTYPE population SYSTEM "http://www.matsim.org/files/dtd/population_v6.dtd">'

with open(output_path, "wb") as f:
    f.write(b'<?xml version="1.0" encoding="UTF-8"?>\n')
    f.write(bytes(dtd_str + '\n', encoding='utf-8'))
    f.write(ET.tostring(population, pretty_print=True, encoding='utf-8'))

print(f"Generated MATSim population file: {output_path}")
